# Phase 2: Text Generation with LLMs

**Goal:** Understand how decoder-only models (GPT-2) generate text, and build intuition for how generation strategies affect output quality.

You now know:
- Text → tokens → IDs → embeddings (Phase 1b)
- Self-attention builds contextual representations (Phase 1)

**This phase:** What happens at the *output* end — how does the model pick the next word?

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Load GPT-2 (small, 124M parameters)
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()

print(f"Model: GPT-2 ({sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters)")
print(f"Vocab size: {tokenizer.vocab_size:,}")

---
## 1. How Does a Language Model Generate Text?

A decoder-only model predicts **one token at a time**. At each step:
1. Feed in all tokens so far
2. Model outputs a probability for **every token in the vocabulary**
3. Pick one token based on those probabilities
4. Add it to the sequence, repeat

Let's see this in slow motion.

In [ ]:
# Step by step: what the model actually outputs

prompt = "The capital of France is"
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

# The model outputs "logits" — raw scores for every token in vocab
logits = outputs.logits
print(f"Input: '{prompt}'")
print(f"Input token IDs: {inputs['input_ids'][0].tolist()}")
print(f"\nLogits shape: {logits.shape}")
print(f"  → 1 batch × {logits.shape[1]} positions × {logits.shape[2]:,} vocab size")
print(f"\nAt EACH position, the model predicts probabilities for ALL {logits.shape[2]:,} tokens.")
print(f"We care about the LAST position — that's the prediction for the next token.")

In [ ]:
# Look at the top predictions for the next token

next_token_logits = logits[0, -1, :]  # last position
probs = F.softmax(next_token_logits, dim=-1)

# Top 15 predictions
top_k = 15
top_probs, top_ids = torch.topk(probs, top_k)

print(f"Prompt: '{prompt}'")
print(f"\nTop {top_k} predictions for the next token:\n")
print(f"{'Rank':<6} {'Token':<15} {'Probability':<15} {'Visual'}")
print("-" * 60)

for rank, (prob, token_id) in enumerate(zip(top_probs, top_ids), 1):
    token = tokenizer.decode([token_id])
    bar = "█" * int(prob.item() * 80)
    print(f"{rank:<6} '{token}'{'':>{12-len(token)}} {prob.item():<15.4f} {bar}")

remaining_prob = 1.0 - top_probs.sum().item()
print(f"\nRemaining {tokenizer.vocab_size - top_k:,} tokens share {remaining_prob:.4f} probability")
print(f"\nThe model is quite confident 'Paris' comes next — but HOW we pick from")
print(f"these probabilities is the 'generation strategy', and it matters a lot.")

### YOUR UNDERSTANDING

*The model doesn't output a word — it outputs probabilities over the entire vocabulary. Why is choosing how to pick from these probabilities important?*

> (your notes here)


---
## 2. Greedy Search: Always Pick the Most Likely Token

The simplest strategy: at each step, pick the token with the **highest probability**.

It's fast and deterministic (same input → same output every time). But it has problems.

In [ ]:
# Implement greedy generation from scratch

def generate_greedy(prompt, max_new_tokens=40):
    """Pick the highest-probability token at each step."""
    input_ids = tokenizer.encode(prompt, return_tensors="pt")
    generated = input_ids.clone()
    
    print(f"Prompt: '{prompt}'\n")
    print(f"{'Step':<6} {'Chosen Token':<15} {'Probability':<12} {'2nd Best':<15} {'2nd Prob'}")
    print("-" * 65)
    
    for step in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(generated)
        
        next_logits = outputs.logits[0, -1, :]
        probs = F.softmax(next_logits, dim=-1)
        
        top2_probs, top2_ids = torch.topk(probs, 2)
        chosen_id = top2_ids[0].unsqueeze(0).unsqueeze(0)
        
        chosen_token = tokenizer.decode([top2_ids[0]])
        second_token = tokenizer.decode([top2_ids[1]])
        
        if step < 15:  # only print first 15 steps
            print(f"{step:<6} '{chosen_token}'{'':>{12-len(chosen_token)}} {top2_probs[0].item():<12.4f} '{second_token}'{'':>{12-len(second_token)}} {top2_probs[1].item():.4f}")
        
        generated = torch.cat([generated, chosen_id], dim=-1)
        
        if top2_ids[0].item() == tokenizer.eos_token_id:
            break
    
    if max_new_tokens > 15:
        print("  ...")
    print(f"\nFull output: '{tokenizer.decode(generated[0])}'")
    return tokenizer.decode(generated[0])

_ = generate_greedy("The future of artificial intelligence is")

In [ ]:
# The problem with greedy: repetition

print("=" * 60)
print("GREEDY SEARCH — Notice the repetition")
print("=" * 60)

prompts = [
    "Once upon a time, there was a",
    "The best way to learn programming is",
    "Customer support should always",
]

for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=False,  # greedy
        )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"\nPrompt: '{prompt}'")
    print(f"Output: '{text}'")
    print()

print("^ Greedy often falls into loops — once it picks a phrase,")
print("  that phrase increases the probability of repeating itself.")

---
## 3. Beam Search: Explore Multiple Paths

Instead of committing to the single best token at each step, **beam search** keeps track of the top N sequences ("beams") and expands all of them.

Think of it like chess: instead of always making the best-looking move, you consider the top 3 moves and think a few steps ahead for each.

In [ ]:
# Beam search with different beam widths

prompt = "The future of artificial intelligence is"
inputs = tokenizer(prompt, return_tensors="pt")

print(f"Prompt: '{prompt}'\n")

for num_beams in [1, 3, 5, 10]:
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=50,
            num_beams=num_beams,
            do_sample=False,
            no_repeat_ngram_size=2,  # prevent repeating 2-word phrases
            early_stopping=True,
        )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    label = "(greedy)" if num_beams == 1 else ""
    print(f"Beams={num_beams:<3} {label}")
    print(f"  {text}\n")

print("More beams = explores more possibilities = often better phrasing.")
print("But still deterministic — same input always gives same output.")
print("And it's slower: beams=5 does 5x the work of greedy.")

### YOUR UNDERSTANDING

*What's the trade-off between greedy search and beam search?*

> (your notes here)


---
## 4. Sampling: Add Randomness

Both greedy and beam search are **deterministic** — same input, same output. That makes text feel robotic and repetitive.

**Sampling** introduces randomness: instead of always picking the top token, we **randomly sample** from the probability distribution. Higher-probability tokens are more likely to be chosen, but lower-probability ones have a chance too.

This is what makes ChatGPT give slightly different answers each time.

In [ ]:
# Pure sampling: run the same prompt 5 times, get different outputs

prompt = "The best recipe for happiness is"
inputs = tokenizer(prompt, return_tensors="pt")

print(f"Prompt: '{prompt}'")
print(f"\nPure sampling (5 runs of the same prompt):\n")

for i in range(5):
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=30,
            do_sample=True,
            top_k=0,  # no filtering — pure sampling from full distribution
        )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"  Run {i+1}: {text}\n")

print("Each run is different! But some may be incoherent because we're")
print("sampling from ALL 50,257 tokens — even very unlikely ones.")

---
## 5. Temperature: Controlling Randomness

**Temperature** controls how "sharp" or "flat" the probability distribution is:
- **Low temperature (0.1-0.3):** Makes high-probability tokens even more dominant → more focused, predictable
- **Temperature = 1.0:** Original distribution (unchanged)
- **High temperature (1.5-2.0):** Flattens probabilities → more random, creative, potentially chaotic

In [ ]:
# Visualize how temperature changes the probability distribution

prompt = "The meaning of life is"
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits[0, -1, :]  # next-token logits

temperatures = [0.2, 0.5, 1.0, 1.5, 2.0]
top_n = 10

fig, axes = plt.subplots(1, len(temperatures), figsize=(18, 4), sharey=True)

for ax, temp in zip(axes, temperatures):
    scaled_logits = logits / temp
    probs = F.softmax(scaled_logits, dim=-1)
    top_probs, top_ids = torch.topk(probs, top_n)
    
    tokens = [tokenizer.decode([id]).strip() for id in top_ids]
    ax.barh(range(top_n-1, -1, -1), top_probs.numpy(), color='steelblue', edgecolor='black')
    ax.set_yticks(range(top_n-1, -1, -1))
    ax.set_yticklabels(tokens, fontsize=8)
    ax.set_title(f'Temp = {temp}', fontsize=11)
    ax.set_xlim(0, 1)

axes[0].set_ylabel('Token')
fig.suptitle(f'Effect of Temperature on Next-Token Probabilities\nPrompt: "{prompt}"', fontsize=12)
plt.tight_layout()
plt.show()

print("Low temp (0.2):  Top token dominates → almost like greedy")
print("Temp 1.0:        Original distribution")
print("High temp (2.0): Probabilities spread out → more randomness")

In [ ]:
# See the effect on actual generated text

prompt = "In a world where robots could dream, they would"
inputs = tokenizer(prompt, return_tensors="pt")

print(f"Prompt: '{prompt}'\n")

for temp in [0.2, 0.5, 0.7, 1.0, 1.5]:
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=True,
            temperature=temp,
            top_k=50,
        )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    label = "(focused)" if temp < 0.5 else "(balanced)" if temp <= 1.0 else "(creative)"
    print(f"Temp={temp:<4} {label}")
    print(f"  {text}\n")

print("This is why ChatGPT has a temperature setting:")
print("  - Code generation → low temp (precise, predictable)")
print("  - Creative writing → high temp (diverse, surprising)")
print("  - General chat    → medium temp (~0.7, balanced)")

### YOUR UNDERSTANDING

*In your own words, what does temperature control and when would you use low vs high?*

> (your notes here)


---
## 6. Top-K Sampling: Limit the Candidates

Pure sampling from all 50K tokens is risky — even tokens with 0.001% probability can get picked, leading to nonsense.

**Top-K sampling:** Only consider the top K tokens, set everything else to zero probability, then sample from those K candidates.

In [ ]:
# Visualize top-k filtering

prompt = "The scientist discovered that"
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits[0, -1, :]
probs = F.softmax(logits, dim=-1)
sorted_probs, sorted_ids = torch.sort(probs, descending=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, k, color in zip(axes, [5, 20, 100], ['#e74c3c', '#3498db', '#2ecc71']):
    top_probs = sorted_probs[:k].numpy()
    remaining = sorted_probs[k:].sum().item()
    
    ax.bar(range(k), top_probs, color=color, alpha=0.7, edgecolor='black', linewidth=0.5)
    ax.set_title(f'Top-K = {k}\n({remaining:.1%} probability discarded)', fontsize=11)
    ax.set_xlabel('Token rank')
    ax.set_ylabel('Probability')
    ax.set_xlim(-1, max(k, 20))

plt.suptitle(f'Top-K Sampling: Only sample from the top K tokens', fontsize=12)
plt.tight_layout()
plt.show()

print("K=5:   Very restricted → safe but potentially boring")
print("K=20:  Moderate → good balance for most tasks")
print("K=100: Permissive → more variety but occasional odd choices")

In [ ]:
# Compare top-k values on the same prompt

prompt = "The secret to great customer service is"
inputs = tokenizer(prompt, return_tensors="pt")

print(f"Prompt: '{prompt}'\n")

for k in [3, 10, 50, 200]:
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=True,
            top_k=k,
            temperature=0.8,
        )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"K={k:<4} → {text}\n")

print("The problem with top-k: a fixed K doesn't adapt to the situation.")
print("Sometimes the model is confident (top 3 tokens hold 95% probability)")
print("and sometimes it's uncertain (probability spread across 100+ tokens).")
print("A fixed K can be too restrictive OR too permissive depending on context.")

---
## 7. Top-P (Nucleus) Sampling: Adaptive Filtering

**Top-P** solves the fixed-K problem. Instead of picking a fixed number of tokens, it picks the **smallest set of tokens whose combined probability reaches P**.

- If the model is confident → only a few tokens are needed → narrow selection
- If the model is uncertain → many tokens needed to reach P → wider selection

It **adapts** to the model's confidence at each step.

In [ ]:
# Visualize how top-p adapts

# Two scenarios: one where model is confident, one where it's uncertain
confident_prompt = "The capital of France is"
uncertain_prompt = "The meaning of life is"

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, prompt, title in zip(axes,
    [confident_prompt, uncertain_prompt],
    ["Confident prediction", "Uncertain prediction"]
):
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    
    logits = outputs.logits[0, -1, :]
    probs = F.softmax(logits, dim=-1)
    sorted_probs, sorted_ids = torch.sort(probs, descending=True)
    
    cumulative = torch.cumsum(sorted_probs, dim=0)
    
    # How many tokens needed to reach p=0.9?
    n_tokens_90 = (cumulative < 0.9).sum().item() + 1
    
    n_show = min(30, len(sorted_probs))
    
    colors = ['#2ecc71' if i < n_tokens_90 else '#e0e0e0' for i in range(n_show)]
    ax.bar(range(n_show), sorted_probs[:n_show].numpy(), color=colors, edgecolor='black', linewidth=0.5)
    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.set_title(f'{title}\n"{prompt}"\nTop-p=0.9 selects {n_tokens_90} tokens (green)', fontsize=10)
    ax.set_xlabel('Token rank')
    ax.set_ylabel('Probability')
    
    top_token = tokenizer.decode([sorted_ids[0]])
    ax.annotate(f'#{1}: "{top_token}" ({sorted_probs[0]:.2%})',
                xy=(0, sorted_probs[0]), xytext=(5, sorted_probs[0]),
                fontsize=8, ha='left')

plt.suptitle('Top-P Sampling Adapts to Model Confidence', fontsize=12)
plt.tight_layout()
plt.show()

print("Left: Model is confident → top-p selects fewer tokens (narrow)")
print("Right: Model is uncertain → top-p selects more tokens (wide)")
print("\nThis adaptation is why top-p is more popular than top-k in practice.")

In [ ]:
# Compare top-p values

prompt = "A good morning routine starts with"
inputs = tokenizer(prompt, return_tensors="pt")

print(f"Prompt: '{prompt}'\n")

for p in [0.3, 0.5, 0.9, 0.95]:
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=True,
            top_p=p,
            top_k=0,  # disable top-k to isolate top-p effect
            temperature=0.8,
        )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"p={p:<5} → {text}\n")

print("Common values in production:")
print("  p=0.9 or p=0.95 — the sweet spot for most applications")
print("  This is what ChatGPT and most LLM APIs default to.")

### YOUR UNDERSTANDING

*What's the advantage of top-p over top-k?*

> (your notes here)


---
## 8. Putting It All Together: Strategy Comparison

Let's compare all strategies side by side on the same prompt.

In [ ]:
# Side-by-side comparison of all strategies

prompt = "The most important lesson I learned in life is that"
inputs = tokenizer(prompt, return_tensors="pt")

strategies = {
    "Greedy": dict(do_sample=False),
    "Beam (5)": dict(do_sample=False, num_beams=5, no_repeat_ngram_size=2, early_stopping=True),
    "Top-K (50)": dict(do_sample=True, top_k=50, temperature=0.8),
    "Top-P (0.9)": dict(do_sample=True, top_p=0.9, top_k=0, temperature=0.8),
    "Top-P (0.9) + Temp 0.3": dict(do_sample=True, top_p=0.9, top_k=0, temperature=0.3),
    "Top-P (0.9) + Temp 1.5": dict(do_sample=True, top_p=0.9, top_k=0, temperature=1.5),
}

print(f"Prompt: '{prompt}'\n")
print("=" * 80)

for name, params in strategies.items():
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=50,
            **params,
        )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    generated_part = text[len(prompt):]
    print(f"\n{name}:")
    print(f"  ...{generated_part}")

print("\n" + "=" * 80)
print("\nNotice how each strategy produces different qualities of text.")
print("There's no 'best' strategy — it depends on your use case.")

In [ ]:
# YOUR TURN: Experiment!
# Try different prompts and settings. Edit the prompt and parameters below.

my_prompt = "Customer support agents should"  # ← change this!
my_inputs = tokenizer(my_prompt, return_tensors="pt")

with torch.no_grad():
    output = model.generate(
        **my_inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.7,      # ← try 0.2, 0.5, 1.0, 1.5
        top_p=0.9,            # ← try 0.5, 0.9, 0.95
        top_k=50,             # ← try 10, 50, 200, or 0 to disable
    )

text = tokenizer.decode(output[0], skip_special_tokens=True)
print(f"Output: {text}")

---
## 9. Summary: Generation Strategy Cheat Sheet

| Strategy | How it works | Pros | Cons | Use when |
|---|---|---|---|---|
| **Greedy** | Always pick top-1 | Fast, deterministic | Repetitive, boring | Quick baseline |
| **Beam Search** | Track top-N paths | Better than greedy, still deterministic | Slow, still repetitive | Translation, summarization |
| **Top-K** | Sample from top K tokens | Adds variety | Fixed K doesn't adapt | Simple generation |
| **Top-P** | Sample from smallest set reaching P% | Adapts to confidence | Slightly more complex | Most production use |
| **Temperature** | Scales probability sharpness | Fine control over randomness | Too high → nonsense | Combined with top-p/top-k |

**Production recipe (what ChatGPT-like systems use):**
```
top_p = 0.9, temperature = 0.7, top_k = 0 (disabled)
```

### YOUR UNDERSTANDING — Final Reflection

*If you were building a customer support chatbot, which generation strategy would you use and why?*

> (your notes here)

*Why is pure greedy search bad for open-ended generation but fine for factual questions?*

> (your notes here)

*What surprised you most about how text generation works?*

> (your notes here)


---
## Next Up: Phase 3 — Prompt Engineering

Now you know how models generate text. The next question is: **how do you control what they generate?**

Phase 3 covers zero-shot, one-shot, and few-shot prompting — how to steer the model's output by crafting the right input, with a practical dialogue summarization task using FLAN-T5.

**Before moving on**, fill in all `YOUR UNDERSTANDING` sections and play with the experiment cell above!